In [ ]:
from google.colab import drive
import pandas as pd

# Mount Google Drive to access the dataset
drive.mount('/content/drive')

# Define the file path provided
file_path = '/content/drive/MyDrive/Colab Notebooks/Datasets/Real estate valuation data set.xlsx'

# Load the Excel file into a pandas DataFrame
df = pd.read_excel(file_path)

# Drop the 'No' column as it is just an index
df = df.drop(columns=['No'])

# Display the data structure to confirm we have the 6 features and 1 target
display(df.head())

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# 1. Separate Features (X) and Target (y)
X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values.reshape(-1, 1)

# 2. Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# 3. Scale the data
scaler_X = StandardScaler()
scaler_y = StandardScaler()

# Fit and transform training data   . find out the mean and variance
X_train_scaled = scaler_X.fit_transform(X_train)
y_train_scaled = scaler_y.fit_transform(y_train)

# Transform testing data using training parameters
X_test_scaled = scaler_X.transform(X_test)  #no "fit" here.
y_test_scaled = scaler_y.transform(y_test)

print("Data Prepared!")
print(f"Training shapes: X={X_train_scaled.shape}, y={y_train_scaled.shape}")
print(f"Testing shapes:  X={X_test_scaled.shape}, y={y_test_scaled.shape}")

In [ ]:
# Hidden Layer Activations
def leaky_relu(z, alpha=0.01):           #Forward Pass
    return np.where(z > 0, z, z * alpha)

def leaky_relu_derivative(a, alpha=0.01):    #Backward Pass
    return np.where(a > 0, 1.0, alpha)

# Output Layer Activations
def linear(z):
    return z

def linear_derivative(output):
    return 1.0


    #it perform : multiplying by 1.0 means the error is passed backward exactly as it is

# 6 -> 8 -> 1 Architecture
input_size = X_train_scaled.shape[1]
hidden_size = 16
output_size = 1

np.random.seed(42)

# He Initialization  : generates a matrix of numbers from a "normal distribution"
W1 = np.random.randn(input_size, hidden_size) * np.sqrt(2 / input_size)
b1 = np.zeros((1, hidden_size))

W2 = np.random.randn(hidden_size, output_size) * np.sqrt(2 / hidden_size)
b2 = np.zeros((1, output_size))

print("Network Ready: 6->8->1 Architecture with He Initialization.")

In [ ]:
def train_one_sample_numpy(x_sample, y_target, lr):
    global W1, b1, W2, b2

    # ------------------- FORWARD PASS -------------------
    Z1 = np.dot(x_sample, W1) + b1
    A1 = leaky_relu(Z1)       # Hidden layer = Leaky ReLU

    Z2 = np.dot(A1, W2) + b2
    predicted_y = linear(Z2)  # Output layer = Linear

    loss = np.mean((predicted_y - y_target) ** 2)

    # ------------------ BACKWARD PASS ------------------
    # Output derivative uses linear_derivative
    d_o1 = 2 * (predicted_y - y_target)   #Error cal

    dW2 = np.dot(A1.T, d_o1)   #Gradients
    db2 = d_o1                  #BIAS

    dA1 = np.dot(d_o1, W2.T)
    # Hidden derivative uses ReLU slope.
    dZ1 = dA1 * leaky_relu_derivative(Z1)

    dW1 = np.dot(x_sample.T, dZ1)   #Gradients for the Input Layer
    db1 = dZ1

    # ---------------- UPDATE PARAMETERS ----------------
    W1 -= lr * dW1
    b1 -= lr * db1

    W2 -= lr * dW2
    b2 -= lr * db2

    return predicted_y, loss

def predict_numpy(x_input):
    Z1 = np.dot(x_input, W1) + b1
    A1 = leaky_relu(Z1)       # Hidden layer = Leaky ReLU
    Z2 = np.dot(A1, W2) + b2
    return linear(Z2)         # Output layer = Linear

print("Training Logic Ready: Gradient math corrected.")

In [ ]:
epochs = 100
learning_rate = 0.005

print("Starting Training with Shuffled Gradient Descent...")

for epoch in range(epochs):
    total_loss = 0.0

    # Shuffle the training data indices every epoch
    indices = np.random.permutation(len(X_train_scaled))

    for i in indices:
        x_sample = X_train_scaled[i:i+1]
        y_target = y_train_scaled[i:i+1]


        #single house> predict>loss
        pred, loss = train_one_sample_numpy(x_sample, y_target, learning_rate)
        total_loss += loss

    avg_loss = total_loss / len(X_train_scaled)

    if epoch % 10 == 0:
        print(f"Epoch {epoch:4d} | Average Training Loss (MSE): {avg_loss:.6f}")

print("\n--- Final Model Evaluation on Unseen Test Data ---")

total_absolute_error = 0.0
total_squared_error = 0.0
predictions_within_tolerance = 0
tolerance_threshold = 0.15

# Lists to store values for R² calculation
real_prices = []
pred_prices = []

for i in range(len(X_test_scaled)):
    x_sample = X_test_scaled[i:i+1]
    real_y_scaled = y_test_scaled[i:i+1]

    pred_y_scaled = predict_numpy(x_sample)

    real_price = scaler_y.inverse_transform(real_y_scaled)[0][0]
    pred_price = scaler_y.inverse_transform(pred_y_scaled)[0][0]

    real_prices.append(real_price)
    pred_prices.append(pred_price)

    error = real_price - pred_price
    absolute_error = abs(error)

    total_absolute_error += absolute_error
    total_squared_error += (error ** 2)

    if absolute_error <= (tolerance_threshold * real_price):
        predictions_within_tolerance += 1

# Manual Math for the Final Metrics
N = len(X_test_scaled)
mae = total_absolute_error / N
rmse = np.sqrt(total_squared_error / N)
accuracy = (predictions_within_tolerance / N) * 100

# Manual R² (Coefficient of Determination)
mean_real_price = np.mean(real_prices)
ss_total = sum((r - mean_real_price)**2 for r in real_prices)
ss_residual = sum((r - p)**2 for r, p in zip(real_prices, pred_prices))
r2_score = (1 - (ss_residual / ss_total)) * 100    #Variance

print(f"MAE  (Mean Absolute Error) : ± {mae:.2f} per unit")
print(f"RMSE (Root Mean Squared)   : ± {rmse:.2f} per unit")
print(f"R²   (Variance Explained)  : {r2_score:.1f}%")
print(f"15% Tolerance Accuracy     : {accuracy:.1f}% of predictions were within 15% of the actual house price.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Your exact setup from the prompt ---
# (Mocking input size as 4 for illustration purposes)
input_size = 6
hidden_size = 8
output_size = 1

layer_sizes = [input_size, hidden_size, output_size]
layer_names = ["Input Layer", f"Hidden Layer\n(Leaky ReLU)", f"Output Layer\n(Linear)"]

# --- Plotting Configuration ---
fig, ax = plt.subplots(figsize=(10, 6))
ax.axis('off')

# Establish horizontal distribution of layers
x_positions = [1, 2, 3]
max_nodes = max(layer_sizes)

# Store node coordinates for drawing weight connections
node_coords = []

# Draw Nodes
for layer_idx, size in enumerate(layer_sizes):
    layer_coords = []
    # Centering nodes vertically on the canvas
    y_positions = np.linspace(-size/2, size/2, size) if size > 1 else [0]

    for node_idx, y in enumerate(y_positions):
        x = x_positions[layer_idx]
        layer_coords.append((x, y))

        # Color coding: Input=Blue, Hidden=Green, Output=Red
        color = 'skyblue' if layer_idx == 0 else ('lightgreen' if layer_idx == 1 else 'salmon')

        # Draw node circle
        circle = plt.Circle((x, y), radius=0.15, color=color, ec='black', zorder=3)
        ax.add_artist(circle)

        # Add labels inside or next to nodes
        node_label = f"X{node_idx+1}" if layer_idx == 0 else (f"H{node_idx+1}" if layer_idx == 1 else "Y")
        ax.text(x, y, node_label, ha='center', va='center', fontweight='bold', zorder=4)

    node_coords.append(layer_coords)
    # Add Layer Titles
    ax.text(x_positions[layer_idx], (max_nodes/2) + 0.3, layer_names[layer_idx],
            ha='center', va='bottom', fontsize=11, fontweight='bold')

# Draw Connections (Weights W1 and W2)
# Connect Input -> Hidden (W1)
for src in node_coords[0]:
    for dest in node_coords[1]:
        ax.annotate("", xy=dest, xytext=src,
                    arrowprops=dict(arrowstyle="->", color="gray", lw=1, shrinkA=15, shrinkB=15))

# Connect Hidden -> Output (W2)
for src in node_coords[1]:
    for dest in node_coords[2]:
        ax.annotate("", xy=dest, xytext=src,
                    arrowprops=dict(arrowstyle="->", color="gray", lw=1, shrinkA=15, shrinkB=15))

# Display limits adjustment
ax.set_xlim(0.5, 3.5)
ax.set_ylim(-max_nodes/2 - 0.5, max_nodes/2 + 0.8)
plt.title("Custom Neural Network Forward Structure", fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()
